In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import torch
from torch import nn
from timeit import default_timer as timer

torch.use_deterministic_algorithms(True)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
data = pd.read_csv('diabetes2.csv')
data.info()

X = data.drop('Outcome', axis=1)
y = data['Outcome']

# Data splitting 60% training, 20% validation, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.4,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.5,
    random_state=42
)

Data_train = pd.concat([X_train, y_train], axis=1)

#spearman correlation (Feature Selection)
Spearman_corr = Data_train.corr(method='spearman')

print(
    Spearman_corr['Outcome']
    .abs()
    .sort_values(ascending=False)
)

number_of_features = 5

Interesting_features = (
    Spearman_corr['Outcome']
    .abs()
    .sort_values(ascending=False)
    .index[1:number_of_features + 1]
)

Interesting_features = Interesting_features.tolist()

X_train = X_train[Interesting_features]
X_val = X_val[Interesting_features]
X_test = X_test[Interesting_features]

# Normalization
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_trained_scaled_frame = {}

for i in range(len(X_train.columns)):
    X_trained_scaled_frame[X_train.columns[i]] = X_train_scaled[:, i]

X_trained_scaled_frame = pd.DataFrame(
    X_trained_scaled_frame,
    index=X_train.index
)

# Balancing
positive_class = np.sum(y_train.values == 1)
negative_class = np.sum(y_train.values == 0)

print(
    f'Positive class: {positive_class}, '
    f'Negative class: {negative_class}'
)

X_train_balanced = pd.concat(
    [
        X_trained_scaled_frame[y_train == 0],
        X_trained_scaled_frame[y_train == 1].sample(
            negative_class,
            replace=True,
            random_state=42
        )
    ],
    axis=0
)

y_train_balanced = pd.concat(
    [
        y_train[y_train == 0],
        y_train[y_train == 1].sample(
            negative_class,
            replace=True,
            random_state=42
        )
    ],
    axis=0
)

print(
    f'Balanced Positive class: '
    f'{np.sum(y_train_balanced.values == 1)}, '
    f'Balanced Negative class: '
    f'{np.sum(y_train_balanced.values == 0)}'
)

X_train_balanced = X_train_balanced.values
y_train_balanced = y_train_balanced.values
y_val = y_val.values
y_test = y_test.values

In [ ]:
class ClassificationModel(nn.Module):

    def __init__(
        self,
        number_of_features,
        hidden_layer_size1,
        hidden_layer_size2,
        hidden_layer_size3,
        number_of_outputs
    ):
        super().__init__()

        self.layer_input = nn.Linear(
            number_of_features,
            hidden_layer_size1
        )  # w*x+b

        self.hidden_layer_1 = nn.Linear(
            hidden_layer_size1,
            hidden_layer_size2
        )

        self.hidden_layer_2 = nn.Linear(
            hidden_layer_size2,
            hidden_layer_size3
        )

        self.layer_output = nn.Linear(
            hidden_layer_size3,
            number_of_outputs
        )

        self.tanh = nn.Tanh()
        # tanh(x) = (e^x - e^-x)/(e^x + e^-x)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        x = self.tanh(self.layer_input(x))
        x = self.tanh(self.hidden_layer_1(x))
        x = self.tanh(self.hidden_layer_2(x))

        return self.layer_output(x)


def train_step(
    model,
    loss_fn,
    X_train,
    y_train,
    optimizer,
    device=device
):

    X_train = torch.tensor(
        X_train,
        dtype=torch.float32
    ).to(device)

    y_train = torch.tensor(
        y_train,
        dtype=torch.float32
    ).to(device)

    model.train()

    y_logist = model(X_train).squeeze()

    y_pred = torch.round(
        torch.sigmoid(y_logist)
    )

    train_loss = loss_fn(
        y_logist,
        y_train
    )

    train_acc = (
        (y_pred == y_train)
        .sum()
        .item()
        / len(y_pred)
    )

    optimizer.zero_grad()

    train_loss.backward()

    optimizer.step()

    return train_loss.item(), train_acc


def val_step(
    model,
    X_val,
    y_val,
    loss_fn,
    optimizer,
    device=device
):

    X_val = torch.tensor(
        X_val,
        dtype=torch.float32
    ).to(device)

    y_val = torch.tensor(
        y_val,
        dtype=torch.float32
    ).to(device)

    model.eval()

    with torch.inference_mode():

        val_logits = model(X_val).squeeze()

        val_pred = torch.round(
            torch.sigmoid(val_logits)
        )

        val_loss = loss_fn(
            val_logits,
            y_val
        )

        val_acc = (
            (val_pred == y_val)
            .sum()
            .item()
            / len(val_pred)
        )

    return val_loss.item(), val_acc


def train(
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    optimizer,
    loss_fn,
    epochs,
    device=device
):

    results = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    best_val_loss = torch.inf

    MODEL_PATH = "model_trained/"

    for epoch in range(epochs):

        train_loss, train_acc = train_step(
            model=model,
            X_train=X_train,
            y_train=y_train,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )

        val_loss, val_acc = val_step(
            model=model,
            X_val=X_val,
            y_val=y_val,
            loss_fn=loss_fn,
            optimizer=optimizer,
            device=device
        )

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            MODEL_NAME = "classifier.pth"

            MODEL_SAVE_PATH = MODEL_PATH + MODEL_NAME

            torch.save(
                model.state_dict(),
                MODEL_SAVE_PATH
            )

        print(
            f"Epoch: {epoch} | "
            f"Train loss: {train_loss:.4f} "
            f"Train acc: {train_acc:.4f} | "
            f"Val loss: {val_loss:.4f} "
            f"Val acc: {val_acc:.4f}"
        )

        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["val_loss"].append(val_loss)
        results["val_acc"].append(val_acc)

    return results

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

NUM_EPOCHS = 5000

HIDDEN_LAYER_SIZE1 = 10
HIDDEN_LAYER_SIZE2 = 6
HIDDEN_LAYER_SIZE3 = 2

model = ClassificationModel(
    number_of_features=number_of_features,
    hidden_layer_size1=HIDDEN_LAYER_SIZE1,
    hidden_layer_size2=HIDDEN_LAYER_SIZE2,
    hidden_layer_size3=HIDDEN_LAYER_SIZE3,
    number_of_outputs=1
).to(device)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [ ]:
start_time = timer()

model_results = train(
    model=model,
    X_train=X_train_balanced,
    y_train=y_train_balanced,
    X_val=X_val_scaled,
    y_val=y_val,
    device=device,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=NUM_EPOCHS
)

end_time = timer()

print(
    f"Total training time "
    f"{end_time-start_time:.3f} seconds"
)

In [ ]:
def plot_loss_curves(results):

    train_loss = results["train_loss"]
    test_loss = results["val_loss"]

    train_acc = results["train_acc"]
    test_acc = results["val_acc"]

    epochs = range(len(train_loss))

    plt.figure(figsize=(15, 7))

    plt.subplot(1, 2, 1)

    plt.plot(
        epochs,
        train_loss,
        label="train_loss"
    )

    plt.plot(
        epochs,
        test_loss,
        label="val_loss"
    )

    plt.title("Loss")
    plt.xlabel("Epochs")
    plt.legend()

    plt.subplot(1, 2, 2)

    plt.plot(
        epochs,
        train_acc,
        label="train_acc"
    )

    plt.plot(
        epochs,
        test_acc,
        label="val_acc"
    )

    plt.title("Accuracy")
    plt.xlabel("Epochs")
    plt.legend()

In [ ]:
plot_loss_curves(model_results)

In [ ]:
model = ClassificationModel(
    number_of_features=number_of_features,
    hidden_layer_size1=HIDDEN_LAYER_SIZE1,
    hidden_layer_size2=HIDDEN_LAYER_SIZE2,
    hidden_layer_size3=HIDDEN_LAYER_SIZE3,
    number_of_outputs=1
).to(device)

model

MODEL_PATH = "model_trained/"
MODEL_NAME = "classifier.pth"

MODEL_SAVE_PATH = MODEL_PATH + MODEL_NAME

model.load_state_dict(
    torch.load(f=MODEL_SAVE_PATH)
)

In [ ]:
model.eval()

with torch.inference_mode():

    y_test_logits = model(
        torch.tensor(
            X_test_scaled,
            dtype=torch.float32
        ).to(device)
    ).squeeze()

    y_test_pred = torch.round(
        torch.sigmoid(y_test_logits)
    )

    y_test_pred = y_test_pred.cpu().numpy()

y_test_pred

In [ ]:
print(
    classification_report(
        y_test,
        y_test_pred
    )
)

In [ ]:
print(
    confusion_matrix(
        y_test,
        y_test_pred
    )
)